IMPORTS

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

Load MAY CPI dataset 

In [2]:
cpi_valuesMay = pd.read_csv(
    r"C:\Users\r8885426\OneDrive - FRG\Documents\G4L\CPI_Historic_Values_Zindi_May_23.csv"
)

Clean the CPI data

In [3]:
# Clean column names
cpi_valuesMay.columns = cpi_valuesMay.columns.str.strip()

# Drop Excel artefact columns
cpi_valuesMay = cpi_valuesMay.loc[
    :, ~cpi_valuesMay.columns.str.contains("^Unnamed")
]

# Keep required columns
cpi_valuesMay = cpi_valuesMay[
    ["Month", "Category", "Value", "Percentage Change (From Prior Month)"]
]

# Clean text
cpi_valuesMay["Category"] = cpi_valuesMay["Category"].astype(str).str.strip()

# Convert data types
cpi_valuesMay["Month"] = pd.to_datetime(
    cpi_valuesMay["Month"], dayfirst=True, errors="coerce"
)

cpi_valuesMay["Value"] = pd.to_numeric(
    cpi_valuesMay["Value"], errors="coerce"
)

# Remove invalid rows
cpi_valuesMay = cpi_valuesMay.dropna(
    subset=["Month", "Category", "Value"]
)

# Remove duplicates (important for cumulative files)
cpi_valuesMay = cpi_valuesMay.drop_duplicates(
    subset=["Month", "Category"], keep="last"
)

# Sort for time series
cpi_valuesMay = cpi_valuesMay.sort_values(["Category", "Month"])

Target CPI categories

In [4]:
categories = [
    "Headline_CPI",
    "Food and non-alcoholic beverages",
    "Alcoholic beverages and tobacco",
    "Clothing and footwear",
    "Housing and utilities",
    "Household contents and services",
    "Health",
    "Transport",
    "Communication",
    "Recreation and culture",
    "Education",
    "Restaurants and hotels",
    "Miscellaneous goods and services"
]

forecast function

In [5]:
def dampened_forecast(series, window=6, alpha=0.5):
    series = series.dropna()
    last_value = series.iloc[-1]

    # Safety fallback
    if len(series) < 3:
        return float(last_value)

    y = series.tail(window).values
    X = np.arange(len(y)).reshape(-1, 1)

    model = LinearRegression()
    model.fit(X, y)

    trend_pred = model.predict([[len(y)]])[0]

    # Dampening
    return float(alpha * trend_pred + (1 - alpha) * last_value)

NOWCAST JUNE 

In [6]:
alphas = [0.4, 0.5, 0.6]
june_predictions = {}

for cat in categories:
    series = (
        cpi_valuesMay[cpi_valuesMay["Category"] == cat]
        .sort_values("Month")["Value"]
    )

    preds = [dampened_forecast(series, alpha=a) for a in alphas]
    june_predictions[cat] = round(np.mean(preds), 1)

june_predictions

{'Headline_CPI': np.float64(110.0),
 'Food and non-alcoholic beverages': np.float64(118.5),
 'Alcoholic beverages and tobacco': np.float64(111.2),
 'Clothing and footwear': np.float64(104.2),
 'Housing and utilities': np.float64(104.7),
 'Household contents and services': np.float64(107.8),
 'Health': np.float64(111.2),
 'Transport': np.float64(113.2),
 'Communication': np.float64(99.8),
 'Recreation and culture': np.float64(105.0),
 'Education': np.float64(111.6),
 'Restaurants and hotels': np.float64(110.5),
 'Miscellaneous goods and services': np.float64(110.1)}

Append JUNE as synthetic observation

In [7]:
last_month = cpi_valuesMay["Month"].max()
june_date = last_month + pd.offsets.MonthEnd(1)

june_rows = pd.DataFrame({
    "Month": june_date,
    "Category": list(june_predictions.keys()),
    "Value": list(june_predictions.values())
})

cpi_with_june = pd.concat(
    [cpi_valuesMay[["Month", "Category", "Value"]], june_rows],
    ignore_index=True
)

NOWCAST JULY

In [8]:
july_predictions = {}

for cat in categories:
    series = (
        cpi_with_june[cpi_with_june["Category"] == cat]
        .sort_values("Month")["Value"]
    )

    preds = [dampened_forecast(series, alpha=a) for a in alphas]
    july_predictions[cat] = round(np.mean(preds), 1)

july_predictions

{'Headline_CPI': np.float64(110.4),
 'Food and non-alcoholic beverages': np.float64(119.0),
 'Alcoholic beverages and tobacco': np.float64(111.9),
 'Clothing and footwear': np.float64(104.3),
 'Housing and utilities': np.float64(104.8),
 'Household contents and services': np.float64(108.0),
 'Health': np.float64(111.9),
 'Transport': np.float64(113.9),
 'Communication': np.float64(99.8),
 'Recreation and culture': np.float64(105.3),
 'Education': np.float64(112.8),
 'Restaurants and hotels': np.float64(110.9),
 'Miscellaneous goods and services': np.float64(110.7)}

 Build SUBMISSION FILES

In [9]:
submission_june = pd.DataFrame(
    [
        ("June_headline CPI", june_predictions["Headline_CPI"]),
        ("June_food and non-alcoholic beverages", june_predictions["Food and non-alcoholic beverages"]),
        ("June_alcoholic beverages and tobacco", june_predictions["Alcoholic beverages and tobacco"]),
        ("June_clothing and footwear", june_predictions["Clothing and footwear"]),
        ("June_housing and utilities", june_predictions["Housing and utilities"]),
        ("June_household contents and services", june_predictions["Household contents and services"]),
        ("June_health", june_predictions["Health"]),
        ("June_transport", june_predictions["Transport"]),
        ("June_communication", june_predictions["Communication"]),
        ("June_recreation and culture", june_predictions["Recreation and culture"]),
        ("June_education", june_predictions["Education"]),
        ("June_restaurants and hotels", june_predictions["Restaurants and hotels"]),
        ("June_miscellaneous goods and services", june_predictions["Miscellaneous goods and services"]),
    ],
    columns=["ID", "Value"]
)

submission_june

,ID,Value
0,June_headline CPI,110.0
1,June_food and non-alcoholic beverages,118.5
2,June_alcoholic beverages and tobacco,111.2
3,June_clothing and footwear,104.2
4,June_housing and utilities,104.7
5,June_household contents and services,107.8
6,June_health,111.2
7,June_transport,113.2
8,June_communication,99.8
9,June_recreation and culture,105.0


In [10]:
submission_july = pd.DataFrame(
    [
        ("July_headline CPI", july_predictions["Headline_CPI"]),
        ("July_food and non-alcoholic beverages", july_predictions["Food and non-alcoholic beverages"]),
        ("July_alcoholic beverages and tobacco", july_predictions["Alcoholic beverages and tobacco"]),
        ("July_clothing and footwear", july_predictions["Clothing and footwear"]),
        ("July_housing and utilities", july_predictions["Housing and utilities"]),
        ("July_household contents and services", july_predictions["Household contents and services"]),
        ("July_health", july_predictions["Health"]),
        ("July_transport", july_predictions["Transport"]),
        ("July_communication", july_predictions["Communication"]),
        ("July_recreation and culture", july_predictions["Recreation and culture"]),
        ("July_education", july_predictions["Education"]),
        ("July_restaurants and hotels", july_predictions["Restaurants and hotels"]),
        ("July_miscellaneous goods and services", july_predictions["Miscellaneous goods and services"]),
    ],
    columns=["ID", "Value"]
)

submission_july

,ID,Value
0,July_headline CPI,110.4
1,July_food and non-alcoholic beverages,119.0
2,July_alcoholic beverages and tobacco,111.9
3,July_clothing and footwear,104.3
4,July_housing and utilities,104.8
5,July_household contents and services,108.0
6,July_health,111.9
7,July_transport,113.9
8,July_communication,99.8
9,July_recreation and culture,105.3
